# Extract All Primary Diagnosis Fields

This notebook is based on `total.ipynb`, but instead of extracting only `Code`, `Diagnosis`, `Emergency`, and `Year`, it extracts all available columns from the ten annual NHS primary diagnosis 3-character workbooks.

The output file is `hospital_diagnosis_all_fields.xlsx`. It includes the original fields, metadata columns, and broad age-group columns for further analysis.


In [ ]:
import os
import re
from pathlib import Path

import pandas as pd

DATA_DIR = Path('./data')
OUTPUT_FILE = Path('./hospital_diagnosis_all_fields.xlsx')


## Helper functions

The NHS workbooks use slightly different sheet names and header positions across years. These helper functions make the extraction more robust.


In [ ]:
def find_sheet(sheet_names):
    """Find the Primary Diagnosis 3 Character sheet across naming variations."""
    for name in sheet_names:
        lower_name = name.lower()
        if 'primary' in lower_name and 'diagnosis' in lower_name and '3' in name and ('char' in lower_name or 'character' in lower_name):
            return name
    return None


def find_start_row(file_path, sheet_name):
    """Find the row containing the real table header."""
    preview = pd.read_excel(file_path, sheet_name=sheet_name, nrows=35, header=None)

    for i, row in preview.iterrows():
        row_str = ' '.join(str(x).lower() for x in row if pd.notna(x))
        if 'primary' in row_str and 'diagnosis' in row_str and 'emergency' in row_str:
            return i

    return None


def extract_academic_year(file_path):
    """Extract academic year information from names such as 2012-13."""
    name = Path(file_path).name
    match = re.search(r'(\d{4})-(\d{2})', name)
    if not match:
        raise ValueError(f'Academic year not found in {name}')

    year_start = int(match.group(1))
    year_end = int(str(year_start)[:2] + match.group(2))
    return f'{year_start}-{match.group(2)}', year_start, year_end


In [ ]:
def clean_columns(df):
    """Standardise column names while preserving all meaningful fields."""
    columns = [str(col).strip() for col in df.columns]

    # The first two columns are code and diagnosis description, even when Excel merged headers create Unnamed columns.
    if len(columns) > 0:
        columns[0] = 'Code'
    if len(columns) > 1:
        columns[1] = 'Diagnosis'

    cleaned = []
    for col in columns:
        col = col.replace('Gender unknown', 'Gender Unknown')
        col = col.replace('Other admission method', 'Other Admission Method')
        col = col.replace('FCE Bed Days', 'FCE bed days')
        cleaned.append(col)

    df.columns = cleaned

    # Later workbooks include a second Emergency column under the zero-bed-day-case section.
    rename_map = {
        'Emergency.1': 'Zero bed day emergency',
        'Elective': 'Zero bed day elective',
        'Other': 'Zero bed day other',
    }
    df = df.rename(columns=rename_map)

    # Remove empty Unnamed columns caused by merged cells, but keep all populated fields.
    empty_unnamed_cols = [
        col for col in df.columns
        if str(col).startswith('Unnamed') and df[col].isna().all()
    ]
    if empty_unnamed_cols:
        df = df.drop(columns=empty_unnamed_cols)

    return df


def add_broad_age_groups(df):
    """Add broader age bands while keeping the original detailed age columns."""
    age_groups = {
        'Age 0-14': ['Age 0', 'Age 1-4', 'Age 5-9', 'Age 10-14'],
        'Age 15-44': ['Age 15', 'Age 16', 'Age 17', 'Age 18', 'Age 19', 'Age 20-24', 'Age 25-29', 'Age 30-34', 'Age 35-39', 'Age 40-44'],
        'Age 45-64': ['Age 45-49', 'Age 50-54', 'Age 55-59', 'Age 60-64'],
        'Age 65-84': ['Age 65-69', 'Age 70-74', 'Age 75-79', 'Age 80-84'],
        'Age 85+': ['Age 85-89', 'Age 90+'],
    }

    for group_name, cols in age_groups.items():
        available_cols = [col for col in cols if col in df.columns]
        if available_cols:
            df[group_name] = df[available_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1, min_count=1)

    return df


## Load one year

This function extracts one annual workbook and returns a cleaned table with all available fields.


In [ ]:
def load_year_all_fields(file_path):
    xls = pd.ExcelFile(file_path)
    sheet_name = find_sheet(xls.sheet_names)
    if sheet_name is None:
        raise ValueError(f'Primary Diagnosis 3 Character sheet not found in {file_path}')

    start_row = find_start_row(file_path, sheet_name)
    if start_row is None:
        raise ValueError(f'Header row not found in {file_path}')

    df = pd.read_excel(file_path, sheet_name=sheet_name, skiprows=start_row)
    df = clean_columns(df)
    df = df.dropna(how='all').reset_index(drop=True)

    academic_year, year_start, year_end = extract_academic_year(file_path)
    df['AcademicYear'] = academic_year
    df['YearStart'] = year_start
    df['YearEnd'] = year_end
    df['SourceFile'] = Path(file_path).name
    df['SourceSheet'] = sheet_name

    # Preserve total rows, but label them clearly.
    code_blank = df['Code'].isna() | (df['Code'].astype(str).str.strip() == '')
    diagnosis_is_total = df['Diagnosis'].astype(str).str.strip().str.lower().eq('total')
    df.loc[code_blank & diagnosis_is_total, 'Code'] = 'Total'

    df['RowType'] = 'Diagnosis'
    df.loc[df['Code'].astype(str).str.strip().str.lower().eq('total'), 'RowType'] = 'Total'

    # Keep diagnosis rows and the annual total row. Drop remaining empty header spacer rows.
    df = df[df['Code'].notna() & df['Diagnosis'].notna()].copy()

    text_cols = {'Code', 'Diagnosis', 'AcademicYear', 'SourceFile', 'SourceSheet', 'RowType'}
    for col in df.columns:
        if col not in text_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    df = add_broad_age_groups(df)

    # Put useful metadata first.
    first_cols = ['AcademicYear', 'YearStart', 'YearEnd', 'RowType', 'Code', 'Diagnosis']
    other_cols = [col for col in df.columns if col not in first_cols]
    return df[first_cols + other_cols]


## Extract all years


In [ ]:
all_files = sorted(DATA_DIR.glob('*.xlsx'))
all_files = [file for file in all_files if not file.name.startswith('~$')]

df_list = []
for file_path in all_files:
    print(f'Loading {file_path.name}')
    df_year = load_year_all_fields(file_path)
    print(f'  rows: {len(df_year):,}, columns: {len(df_year.columns):,}')
    df_list.append(df_year)

df_all = pd.concat(df_list, ignore_index=True)
df_all.head()


## Summary tables

These summaries help check the extraction and support later analysis.


In [ ]:
diagnosis_rows = df_all[df_all['RowType'] == 'Diagnosis'].copy()
total_rows = df_all[df_all['RowType'] == 'Total'].copy()

year_summary = (
    diagnosis_rows.groupby(['AcademicYear', 'YearStart'], as_index=False)['Emergency']
    .sum()
    .sort_values('YearStart')
)

top_emergency_diagnoses = (
    diagnosis_rows.groupby(['Code', 'Diagnosis'], as_index=False)['Emergency']
    .sum()
    .sort_values('Emergency', ascending=False)
    .head(20)
)

age_group_cols = ['Age 0-14', 'Age 15-44', 'Age 45-64', 'Age 65-84', 'Age 85+']
available_age_group_cols = [col for col in age_group_cols if col in diagnosis_rows.columns]
age_group_summary = (
    diagnosis_rows.groupby(['AcademicYear', 'YearStart'], as_index=False)[available_age_group_cols]
    .sum()
    .sort_values('YearStart')
) if available_age_group_cols else pd.DataFrame()

sex_summary = (
    diagnosis_rows.groupby(['AcademicYear', 'YearStart'], as_index=False)[['Male', 'Female', 'Gender Unknown']]
    .sum()
    .sort_values('YearStart')
)

year_summary.head(), top_emergency_diagnoses.head()


## Export

The Excel output keeps the full extracted dataset and adds several summary sheets.


In [ ]:
column_dictionary = pd.DataFrame({
    'Column': df_all.columns,
    'Description': [
        'Metadata, diagnosis label, original NHS metric, or derived broad age group.'
        for _ in df_all.columns
    ]
})

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    df_all.to_excel(writer, sheet_name='AllFields', index=False)
    diagnosis_rows.to_excel(writer, sheet_name='DiagnosisRows', index=False)
    total_rows.to_excel(writer, sheet_name='TotalRows', index=False)
    year_summary.to_excel(writer, sheet_name='YearSummary', index=False)
    top_emergency_diagnoses.to_excel(writer, sheet_name='TopEmergencyDiagnoses', index=False)
    age_group_summary.to_excel(writer, sheet_name='AgeGroupSummary', index=False)
    sex_summary.to_excel(writer, sheet_name='SexSummary', index=False)
    column_dictionary.to_excel(writer, sheet_name='ColumnDictionary', index=False)

OUTPUT_FILE
